**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/park-2019"], check=True)
    os.chdir("/content/encore/entries/park-2019")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Earlier findings checked on CRCNS hc-20 (unversioned)

This notebook belongs to the reconciliation of attempt park-2019-02 with attempt park-2019-01
(docs/protocol.md, "Repeat attempts"). Of the twelve findings of park-2019-01, park-2019-02
reports again the same problem for ten. The two it did not report, findings 5 and 12, are checked
here, and so is the part of finding 2 it did not report (the second sheet of the file list). The
data are those of both attempts: CRCNS hc-20, `hc-20/live/data.zip`, unversioned, pinned by the
MD5 sum of `data.zip`, streamed with the entry's helper module `hd20.py`. Each section opens
with the earlier statement.

In [ ]:
import io
import urllib.request
import warnings

import numpy as np
import pandas as pd

import hd20

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200)
sums = hd20.checksums()
print("data.zip MD5 as pinned:", hd20.DATA_ZIP_MD5, "; as published now:", sums["data.zip"],
      "; same:", sums["data.zip"] == hd20.DATA_ZIP_MD5)
z = hd20.open_zip()

## Earlier finding 2 (metadata_error)

> FileNameList.xlsx misaligns its stable2 column for five units of session 1121R157. The rows for
> SA1 P11C2, P11C3, P11C4, P9C2 and P9C3 list SA2 P11C3, P11C4, P9C2, P9C3 and cl1.P11C2, one row
> off. Its second sheet also names the stable2 files of 1020R035 and 1030R035 as CON files, while
> the files are named SA2.

The rows of session 1121R157 in the first sheet, the rows of 1020R035 and 1030R035 in the second
sheet, and the stable2 files of those two sessions in the zip.

In [ ]:
sheets = pd.read_excel(io.BytesIO(z.read("data/TSD/FileNameList.xlsx")), sheet_name=None, header=None,
                       engine="openpyxl")
print("sheets:", list(sheets))
s1 = sheets["Sheet1"].astype(str)
print("\nSheet1, session 1121R157 (SA1 and SA2 columns):")
print(s1[s1[0].str.startswith("1121SA1R157")][[0, 3]].to_string(index=False, header=False))
s2 = sheets["Sheet2"].astype(str)
print("\nSheet2, sessions 1020R035 and 1030R035:")
print(s2[s2[0].str.contains(r"^(1020|1030).*R035")].to_string(index=False, header=False))
names = z.namelist()
print("\nTSD files of 1020R035 and 1030R035 in data/TSD/SA2/:",
      sorted(n.split("/")[-1] for n in names if n.startswith("data/TSD/SA2/") and "R035" in n and n.split("/")[-1][:4] in ("1020", "1030")))
print("file names containing CON anywhere in the zip:", [n for n in names if "CON" in n])

The output above shows whether the shifted stable2 column and the CON names are still there.
The conclusion for this finding is in the reconciliation block of `attempts/park-2019-02.yaml`.

## Earlier finding 5 (format_error)

> In sessions 1009R029, 1015R029, 1023R029 and 1025R029 the room-frame and arena-frame tracks of
> the rotating trial disagree with each other (median 6.9 to 13.5 px in distance from the arena
> centre at the best constant lag) and with the session's .dat files, which agree with each other
> (0.31 px). The lag of the TSD tracks behind the .dat track grows through the trial, to about 700
> to 1500 samples in the room frame and half that in the arena frame, so the spikes cannot be
> aligned with both frames. Excluding these sessions changes the room-frame results of the
> paper's directionality and internally referenced tuning comparisons.

The distance from the arena centre (127.5, 127.5 px) does not change with the arena's rotation,
so it should agree between the two frames of a trial. For each session, from the TSD files of the
first unit the file list gives (the cells of a session share their tracking) and from the room and
arena .dat files of the rotating trial: the median absolute difference of that distance between
the TSD room and arena tracks at the best constant lag (searched over plus and minus 1500 samples),
the same between the two .dat files at lag 0, and the lag of each TSD track against the room .dat
track in 3000-sample segments starting at samples 6000, 24000 and 48000 (searched over plus and
minus 1500 samples; a negative lag means the TSD track runs behind the .dat track). Session
1022R029 is shown for comparison. Positions with x = y = 0 are lost and left out.

In [ ]:
def radius(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    r = np.hypot(x - 127.5, y - 127.5)
    r[(x == 0) & (y == 0)] = np.nan
    return r


def best_lag(a, b):
    best = None
    for L in range(-1500, 1501, 3):
        x, y = (a[L:], b) if L >= 0 else (a, b[-L:])
        n = min(len(x), len(y))
        m = np.nanmedian(np.abs(x[:n] - y[:n]))
        if best is None or m < best[0]:
            best = (m, L)
    return best


def lag_at(a, d, w0, w=3000):
    c = [(np.nanmedian(np.abs(a[w0:w0 + w] - d[w0 + L:w0 + L + w])), L) for L in range(-1500, 1500, 3)
         if 0 <= w0 + L and w0 + L + w <= len(d)]
    return min(c)[1]


fl = hd20.file_list()
dat = hd20.dat_files()
rows = []
for s in ["1009R029", "1015R029", "1023R029", "1025R029", "1022R029"]:
    date, rat = s[:4], s[5:]
    rm = fl.RA_rm[fl.RA_rm.str.startswith(date) & fl.RA_rm.str.contains("R" + rat)].iloc[0]
    ar = fl.RA_ar[fl.RA_ar.str.startswith(date) & fl.RA_ar.str.contains("R" + rat)].iloc[0]
    trm = hd20.parse_tsd(z.read(f"data/TSD/RA_rm/{rm}"))
    tar = hd20.parse_tsd(z.read(f"data/TSD/RA_ar/{ar}"))
    paths = dat[(dat.session == s) & (dat.trial == "RA")].set_index("frame").path
    _, droom = hd20.parse_dat(z.read(paths["room"]))
    _, darena = hd20.parse_dat(z.read(paths["arena"]))
    a_rm, a_ar = radius(trm["x"], trm["y"]), radius(tar["x"], tar["y"])
    r_rm, r_ar = radius(droom.x, droom.y), radius(darena.x, darena.y)
    n = min(len(r_rm), len(r_ar))
    m, L = best_lag(a_rm, a_ar)
    rows.append({"session": s, "TSD room vs arena (px)": round(float(m), 2), "at lag": L,
                 ".dat room vs arena (px)": round(float(np.nanmedian(np.abs(r_rm[:n] - r_ar[:n]))), 2),
                 "TSD room lag at 6000, 24000, 48000": [lag_at(a_rm, r_rm, w) for w in (6000, 24000, 48000)],
                 "TSD arena lag at 6000, 24000, 48000": [lag_at(a_ar, r_rm, w) for w in (6000, 24000, 48000)]})
print(pd.DataFrame(rows).to_string(index=False))

The output above shows whether the two TSD frames of the four sessions still disagree, and
whether their lag against the tracker file still grows through the trial; a lag of -1500 is the
edge of the search. The consequence for the paper's comparisons is not run again here. The
conclusion for this finding is in the reconciliation block of `attempts/park-2019-02.yaml`.

## Earlier finding 12 (paper_inconsistency)

> The decoding window is 660 ms in the STAR Methods (20 samples in the code, 667 ms at 30 Hz) and
> 133 ms in the Figure 2 legend. The arena-frame error z is 289.63 in the Results and 153.24 and
> 230.58 in the Figure 5 legend, the values given for the head-direction distributions.

This finding is about the paper and its code, which do not depend on the data set's version. The
paper's passages, from the version of record: STAR Methods, Decoding head-direction from ensemble
spike trains, "A Maximum Likelihood Estimator was used to determine the head direction of the rat
at a 660 ms time resolution (20 time bins of 33 ms each)"; Figure 2 legend, "using stable1
directional tuning to decode direction each 133 ms during the subsequent trial"; Results, "The
corresponding error distribution in the arena frame (Figure 5B) was also inhomogeneous
(Rayleigh's test; r = 0.653; z = 289.63; p = 0.0001)"; Figure 5 legend, "room frame: resultant
vector r = 0.147, Rayleigh's test z = 230.58, p < 0.0001; arena frame: r = 0.378, z = 153.24,
p < 0.0001" for the head directions, and "The arena frame directional error is inhomogeneous
(resultant vector r = 0.653; Rayleigh's test z = 153.24; z = 230.58; p = 0)". The cell below reads
the decoder's window from the paper's code at the commit the entry ports, and the tracker's
sampling rate from the data set.

In [ ]:
url = ("https://raw.githubusercontent.com/FentonLab/PyCode_MEC_HDC-NeuronMS_Park/545a738a/"
       "HD_Max_likelihood.py")
src = urllib.request.urlopen(url, timeout=60).read().decode()
line = [l for l in src.splitlines() if l.startswith("num_samps")][0]
print("HD_Max_likelihood.py:", line.strip())
num_samps = int(line.split("=")[1].split("#")[0])
fs = hd20.parse_tsd(z.read("data/TSD/SA1/" + fl.SA1.iloc[0]))["fs"]
print(f"tracker rate in {fl.SA1.iloc[0]}: {fs} samples/s; window of {num_samps} samples = {1000 * num_samps / fs:.0f} ms; "
      f"133 ms = {0.133 * fs:.1f} samples")

The paper's text is unchanged, and the cell above shows the window the code uses. The
conclusion for this finding is in the reconciliation block of `attempts/park-2019-02.yaml`.